# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/brentjoel/FlyRank-1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

## 1. Unit of analysis + time window

My lane is still Lane 4, CTR opportunity: pages that get a lot fewer clicks than other pages sitting at a similar Google position. In ML-03 I framed it as scoring and ranking pages for an editor.

**My contract in plain words**

1. **What one row means.** In the warehouse table one row is one content item on one day (`report_date` + `client_hash_id` + `content_hash_id`). For my lane I add those days up so I get one row per page per window, which is the same "one page" unit I used in ML-03.
2. **Which table.** `fact_content_daily_performance`, only the `month=2026-03` partition. I only need the GSC columns (impressions, clicks, summed position), so no join this week.
3. **Which time window.** March 2026, cut in two at a decision moment, the end of 15 March:
   - 1 to 15 March is the feature window (what an editor could know on 15 March)
   - 16 to 31 March is the outcome window (only used to build the label)
   - I don't touch June 2026 / the `_sample` table at all. That stays my sealed test month.
4. **What I predict.** `gap_persists` = 1 if the page's CTR in 16–31 March is below half of the pooled CTR of pages in the same position tier. The tier comes from the feature window. It's the same proxy as ML-03, just on real daily data now. The clicks are observed, the `< 0.5` cut-off is my own choice.
5. **What I leave out on purpose.** All the GA4 columns (sessions, engagement, AI referrals, scroll). My lane is about whether people click the search result, and GA4 is filled with zeros before each client's GA4 start date (`ga4_data_available = FALSE`), so a zero there can mean "not measured" instead of "no engagement". IDs and anything from the outcome window are also never features.

I also use a volume floor: a page needs at least 150 impressions in each half (about the same as the 300 per 30 days I used in ML-03). Below that one click can swing the CTR a lot.

In [1]:
# Setup: DuckDB reading the gated Hugging Face parquet files.
# The token is never written in this notebook. It comes from the HF_TOKEN env var,
# a Colab Secret called HF_TOKEN, or the local huggingface_hub login.
import os, sys, subprocess, time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "duckdb", "huggingface_hub"], check=True)
    if not os.path.isdir("FlyRank-1"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/brentjoel/FlyRank-1"], check=True)
    os.chdir("FlyRank-1")
else:
    here = Path.cwd()
    while not (here / "work").is_dir() and here.parent != here:
        here = here.parent
    os.chdir(here)

import duckdb, numpy as np, pandas as pd

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not HF_TOKEN:
    from huggingface_hub import get_token
    HF_TOKEN = get_token()
assert HF_TOKEN, "No HF token found - add a Colab Secret named HF_TOKEN (or run huggingface_hub.login() locally)."

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
del HF_TOKEN

REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"        # mid-panel month, June 2026 stays sealed
SPLIT = "2026-03-15"     # decision moment = end of this day
MIN_IMPR = 150           # impressions needed in each half
SRC = f"read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')"

# load only the columns I need, once, so I don't keep hitting the remote files
t0 = time.time()
con.execute(f'''
    CREATE OR REPLACE TABLE march AS
    SELECT report_date, client_hash_id, content_hash_id,
           gsc_data_available, gsc_impressions, gsc_clicks, gsc_sum_position
    FROM {SRC}
''')
print(f"loaded month={MONTH} (only my lane's columns) in {time.time() - t0:.0f}s")

loaded month=2026-03 (only my lane's columns) in 56s


## 2. Fields: feature / label / context / excluded

| Bucket | Field(s) | Why |
|---|---|---|
| Context | `report_date`, `client_hash_id`, `content_hash_id`, `month` | dates make the windows; IDs are only for grouping and the split by client, never inputs |
| Context (filter) | `gsc_data_available` | decides which rows count, filtered with `IS TRUE` |
| Feature source | `gsc_impressions`, `gsc_clicks`, `gsc_sum_position` added up over 1–15 March only | already known on the decision day |
| Label / proxy | the same columns added up over 16–31 March → `ctr_gap_ratio_h2` → `gap_persists` | this is the future, never a feature |
| Excluded | `gsc_avg_position` (daily) | it's an average per day, and averaging it again over days ignores impressions. I use `SUM(gsc_sum_position) / SUM(gsc_impressions)` instead |
| Excluded | all `ga4_*`, `sessions_*`, `ai_*`, `scroll_events`, `ga4_data_available`, `client_has_ga4` | not my lane, and zero-filled before each client's GA4 start |
| Excluded | `month=2026-06` / the `_sample` table | sealed test month, it's the natural outcome window for any past→future label |

The next cell is a small guard so a label column or an ID can't sneak into my feature list by accident.

In [2]:
FEATURES = ["log_impr_h1", "pos_h1", "ctr_h1", "ctr_gap_ratio_h1", "active_days_h1"]
LABEL = "gap_persists"
LABEL_SOURCES = ["impr_h2", "clicks_h2", "ctr_h2", "peer_ctr_h2", "ctr_gap_ratio_h2", LABEL]
CONTEXT = ["client_hash_id", "content_hash_id", "report_date", "month", "gsc_data_available"]
EXCLUDED_PREFIXES = ("ga4_", "sessions_", "ai_", "scroll_", "client_has_")

assert not set(FEATURES) & set(LABEL_SOURCES), "a label-window column is in the features"
assert not set(FEATURES) & set(CONTEXT), "an ID / context column is in the features"
assert not any(f.startswith(EXCLUDED_PREFIXES) for f in FEATURES), "an excluded column is in the features"
assert all(f.endswith("_h1") for f in FEATURES), "every feature must come from the 1-15 March window"
print("buckets don't overlap, every feature is from the 1-15 March window (_h1)")
print("features:", FEATURES)
print("label   :", LABEL, "<- built from", LABEL_SOURCES[:-1])

buckets don't overlap, every feature is from the 1-15 March window (_h1)
features: ['log_impr_h1', 'pos_h1', 'ctr_h1', 'ctr_gap_ratio_h1', 'active_days_h1']
label   : gap_persists <- built from ['impr_h2', 'clicks_h2', 'ctr_h2', 'peer_ctr_h2', 'ctr_gap_ratio_h2']


## 3. Verify it with queries (grain, counts, missing values, windows)

Three checks, all on `month=2026-03`.

**Query 1: grain.** If one row really is one day × client × content item, the row count should equal the number of distinct key combinations, with zero duplicates.

In [3]:
q1 = con.sql('''
    SELECT COUNT(*)                                                         AS rows,
           COUNT(DISTINCT (report_date, client_hash_id, content_hash_id))   AS distinct_keys,
           COUNT(*) - COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS duplicate_rows
    FROM march
''').df()
print(q1.to_string(index=False))
assert int(q1.duplicate_rows[0]) == 0, "grain broken"
print("-> grain holds: one row = one content item on one day")

   rows  distinct_keys  duplicate_rows
9841378        9841378               0
-> grain holds: one row = one content item on one day


**Query 2: row count and date span.** I expect 1 to 31 March, so 15 days in my feature half and 16 in my outcome half.

In [4]:
q2 = con.sql(f'''
    SELECT COUNT(*)                                        AS rows,
           MIN(report_date)                                AS first_day,
           MAX(report_date)                                AS last_day,
           COUNT(DISTINCT report_date)                     AS days,
           COUNT(DISTINCT report_date) FILTER (WHERE report_date <= DATE '{SPLIT}') AS days_feature_window,
           COUNT(DISTINCT report_date) FILTER (WHERE report_date >  DATE '{SPLIT}') AS days_outcome_window,
           COUNT(DISTINCT client_hash_id)                  AS clients,
           COUNT(DISTINCT content_hash_id)                 AS content_items
    FROM march
''').df()
print(q2.T.to_string(header=False))

rows                             9841378
first_day            2026-03-01 00:00:00
last_day             2026-03-31 00:00:00
days                                  31
days_feature_window                   15
days_outcome_window                   16
clients                               55
content_items                     331437


**Query 3: availability with `IS TRUE`.** Only rows where `gsc_data_available IS TRUE` should have real search data. The flag can be TRUE, FALSE or NULL, so I count all three. Writing `= TRUE` or `NOT ...` would handle NULLs wrong without any error.

In [5]:
q3 = con.sql('''
    SELECT COUNT(*)                                                  AS all_rows,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)        AS gsc_is_true,
           COUNT(*) FILTER (WHERE gsc_data_available IS FALSE)       AS gsc_is_false,
           COUNT(*) FILTER (WHERE gsc_data_available IS NULL)        AS gsc_is_null,
           ROUND(100.0 * COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) / COUNT(*), 1) AS pct_surviving,
           COALESCE(SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS NOT TRUE), 0) AS impressions_on_dropped_rows
    FROM march
''').df()
print(q3.T.to_string(header=False))

all_rows                     9841378.0
gsc_is_true                  3611061.0
gsc_is_false                 6230317.0
gsc_is_null                        0.0
pct_surviving                     36.7
impressions_on_dropped_rows        0.0


**What I learned from the three queries:** the grain holds (0 duplicates). March has 9.84M rows over exactly 31 days, split 15 + 16. Only about 37% of the rows survive `IS TRUE`, which surprised me at first, but the dropped rows have 0 impressions, so I'm not losing any search data. They're just placeholder rows for pages that had no GSC record that day. There are no NULL flags this month, but `IS TRUE` would still handle them.

### 3b. Five features and when each one is known

One SQL query turns the daily rows into one row per page, using only the `IS TRUE` rows. Peer CTR is the pooled CTR (total clicks / total impressions) of the pages in the same position tier. For the feature version I only use pages that pass the floor in 1–15 March, so the peer group doesn't depend on anything that happens later.

In [6]:
TIER_SQL = '''CASE WHEN pos_h1 <= 3 THEN 'top_3' WHEN pos_h1 <= 10 THEN 'page_1'
                   WHEN pos_h1 <= 20 THEN 'striking' WHEN pos_h1 <= 50 THEN 'page_3_5' ELSE 'deep' END'''

frame = con.sql(f'''
    WITH page AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions)  FILTER (WHERE report_date <= DATE '{SPLIT}') AS impr_h1,
               SUM(gsc_clicks)       FILTER (WHERE report_date <= DATE '{SPLIT}') AS clicks_h1,
               SUM(gsc_sum_position) FILTER (WHERE report_date <= DATE '{SPLIT}') AS sumpos_h1,
               COUNT(*)              FILTER (WHERE report_date <= DATE '{SPLIT}' AND gsc_impressions > 0) AS active_days_h1,
               SUM(gsc_impressions)  FILTER (WHERE report_date >  DATE '{SPLIT}') AS impr_h2,
               SUM(gsc_clicks)       FILTER (WHERE report_date >  DATE '{SPLIT}') AS clicks_h2
        FROM march
        WHERE gsc_data_available IS TRUE
        GROUP BY 1, 2
    ),
    tiered AS (                       -- feature side: only pages with enough data by 15 March
        SELECT *, {TIER_SQL} AS tier_h1
        FROM (SELECT *, sumpos_h1 * 1.0 / impr_h1 AS pos_h1 FROM page WHERE impr_h1 >= {MIN_IMPR})
    ),
    with_peer_h1 AS (                 -- peer CTR uses 1-15 March pages and numbers only
        SELECT *, 100.0 * SUM(clicks_h1) OVER (PARTITION BY tier_h1)
                        / SUM(impr_h1)   OVER (PARTITION BY tier_h1) AS peer_ctr_h1
        FROM tiered
    )
    SELECT client_hash_id, content_hash_id, tier_h1,
           impr_h1, clicks_h1, active_days_h1, pos_h1,
           100.0 * clicks_h1 / impr_h1 AS ctr_h1,
           peer_ctr_h1,
           -- outcome window, label side only (pages we can also measure in 16-31 March)
           impr_h2, clicks_h2,
           100.0 * clicks_h2 / impr_h2 AS ctr_h2,
           100.0 * SUM(clicks_h2) OVER (PARTITION BY tier_h1)
                 / SUM(impr_h2)   OVER (PARTITION BY tier_h1) AS peer_ctr_h2
    FROM with_peer_h1
    WHERE impr_h2 >= {MIN_IMPR}
''').df()

frame["log_impr_h1"] = np.log1p(frame["impr_h1"])
frame["ctr_gap_ratio_h1"] = frame["ctr_h1"] / frame["peer_ctr_h1"]
frame["ctr_gap_ratio_h2"] = frame["ctr_h2"] / frame["peer_ctr_h2"]       # label source
frame[LABEL] = (frame["ctr_gap_ratio_h2"] < 0.5).astype(int)

assert frame["content_hash_id"].is_unique, "feature frame grain broken"
Path("work/outputs").mkdir(parents=True, exist_ok=True)
frame.to_parquet(f"work/outputs/w03_lane4_frame_{MONTH}.parquet", index=False)   # *.parquet is git-ignored

print(f"pages with >= {MIN_IMPR} impressions in BOTH halves: {len(frame):,} "
      f"from {frame['client_hash_id'].nunique()} clients")
print(f"base rate of {LABEL}: {frame[LABEL].mean():.1%}")
print("\npooled peer CTR % by tier (feature window | outcome window):")
print(frame.groupby("tier_h1")[["peer_ctr_h1", "peer_ctr_h2"]].first()
           .reindex(["top_3", "page_1", "striking", "page_3_5", "deep"]).round(3).to_string())
frame[FEATURES + [LABEL]].describe().T.round(3)

pages with >= 150 impressions in BOTH halves: 63,489 from 34 clients
base rate of gap_persists: 49.1%

pooled peer CTR % by tier (feature window | outcome window):
          peer_ctr_h1  peer_ctr_h2
tier_h1                           
top_3           0.437        0.362
page_1          0.330        0.297
striking        0.338        0.296
page_3_5        0.141        0.159
deep            0.039        0.041


,count,mean,std,min,25%,50%,75%,max
log_impr_h1,63489.0,6.815,1.131,5.017,5.911,6.692,7.561,11.993
pos_h1,63489.0,11.728,11.882,0.014,4.054,6.912,15.082,90.400
ctr_h1,63489.0,0.304,0.433,0.000,0.000,0.176,0.435,12.928
ctr_gap_ratio_h1,63489.0,0.997,1.474,0.000,0.000,0.572,1.395,44.781
active_days_h1,63489.0,14.318,1.873,2.000,15.000,15.000,15.000,15.000
gap_persists,63489.0,0.491,0.500,0.000,0.000,0.000,1.000,1.000


**My five features and why each one is known on 15 March**

| # | Feature | Known at the decision moment because… |
|---|---|---|
| 1 | `log_impr_h1`: log of impressions, 1–15 Mar | it only adds up impressions from days up to 15 March |
| 2 | `pos_h1`: position weighted by impressions, 1–15 Mar | it's `SUM(gsc_sum_position) / SUM(gsc_impressions)` over those same days |
| 3 | `ctr_h1`: CTR in %, 1–15 Mar | clicks / impressions from days up to 15 March |
| 4 | `ctr_gap_ratio_h1`: page CTR / peer CTR of its tier, 1–15 Mar | the page CTR, the tier and the peer group all come from 1–15 March data only |
| 5 | `active_days_h1`: days out of 15 with at least 1 impression | it only counts days up to 15 March |

One thing to be honest about: I also need 150+ impressions in 16–31 March, otherwise I can't grade the page. That decides which pages get a label, but none of the features use it.

### 3c. Quick honest score

I split by client (`client_hash_id`), so the test clients are never seen in training. Model is gradient-boosted trees, scored with ROC AUC and precision@50 (my metric from ML-03).

In [7]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
tr_idx, te_idx = next(gss.split(frame, frame[LABEL], groups=frame["client_hash_id"]))
train, test = frame.iloc[tr_idx], frame.iloc[te_idx]
assert not set(train.client_hash_id) & set(test.client_hash_id), "client leak across the split"

def quick_score(cols, name):
    m = HistGradientBoostingClassifier(random_state=0).fit(train[cols], train[LABEL])
    p = m.predict_proba(test[cols])[:, 1]
    top50 = test.assign(p=p).nlargest(50, "p")
    return {"run": name, "n_features": len(cols),
            "roc_auc": round(roc_auc_score(test[LABEL], p), 3),
            "precision@50": round(top50[LABEL].mean(), 3)}

print(f"train: {len(train):,} pages / {train.client_hash_id.nunique()} clients   "
      f"test: {len(test):,} pages / {test.client_hash_id.nunique()} clients   "
      f"test base rate: {test[LABEL].mean():.1%}")
honest = quick_score(FEATURES, "honest: 5 features")
pd.DataFrame([honest])

train: 30,096 pages / 23 clients   test: 33,393 pages / 11 clients   test base rate: 50.7%


,run,n_features,roc_auc,precision@50
0,honest: 5 features,5,0.793,0.98


### 3d. The trap: adding a label column on purpose

Now I add `ctr_gap_ratio_h2`, the CTR / peer CTR from 16–31 March. The label is just `ctr_gap_ratio_h2 < 0.5`, so this column basically is the answer. On 15 March it doesn't exist yet, because it needs clicks from the second half of the month.

In [8]:
LEAK = "ctr_gap_ratio_h2"
leaky = quick_score(FEATURES + [LEAK], f"LEAKY: 5 features + {LEAK}")
pd.DataFrame([honest, leaky])

,run,n_features,roc_auc,precision@50
0,honest: 5 features,5,0.793,0.98
1,LEAKY: 5 features + ctr_gap_ratio_h2,6,1.000,1.00


The score jumps to basically perfect. The model didn't get smarter, it just got to see the future. If I shipped that on 15 March the column wouldn't exist and the score would fall apart.

So I drop the column, check it's really gone, and keep the honest number.

In [9]:
frame = frame.drop(columns=[LEAK])
train, test = frame.iloc[tr_idx], frame.iloc[te_idx]
assert LEAK not in frame.columns and LEAK not in FEATURES

final = quick_score(FEATURES, "FINAL (leak removed)")
result = pd.DataFrame([honest, leaky, final]).set_index("run")
print(f"AUC inflation from one leaked column: +{leaky['roc_auc'] - final['roc_auc']:.3f}")
print(f"honest number I keep: ROC AUC {final['roc_auc']:.3f}, precision@50 {final['precision@50']:.1%} "
      f"(base rate {test[LABEL].mean():.1%})")
result

AUC inflation from one leaked column: +0.207
honest number I keep: ROC AUC 0.793, precision@50 98.0% (base rate 50.7%)


,n_features,roc_auc,precision@50
run,,,
honest: 5 features,5,0.793,0.98
LEAKY: 5 features + ctr_gap_ratio_h2,6,1.000,1.00
FINAL (leak removed),5,0.793,0.98


## 4. Data limits

**My main limitation: the volume floor leaves me with mostly big pages from a few big clients.** To trust a CTR over two 15–16 day halves I only keep pages with 150+ impressions in both halves. The cell below shows what that costs. About two thirds of the pages with search data drop out (but they only have around 5% of the impressions, they're the long tail), and 13 of the 47 clients with search data lose all their pages. Also, five clients hold about 76% of the pages I keep, so the split by client is lumpy: one big client landing in the test set makes the test set bigger than the training set. So my numbers above are directional and only about established, higher-traffic pages of a few large clients. They don't say anything about small or new pages or about the clients that got filtered out.

Other smaller limits: one month means only one split and no seasonality. 15 days is shorter than the 30-day windows from ML-03, so CTR is noisier. GSC data usually comes in 2–3 days late, so on a real 15 March the last few days might not be there yet. Position is an average over all the queries a page shows up for (a few pages even show an average below 1, probably rounding in the summed position). And with one split I don't have an error bar on the score yet.

In [10]:
lim = con.sql('''
    WITH p AS (
        SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS impr
        FROM march WHERE gsc_data_available IS TRUE GROUP BY 1, 2
    )
    SELECT COUNT(*) AS pages_with_search_data,
           COUNT(DISTINCT client_hash_id) AS clients_with_search_data,
           SUM(impr) AS impressions
    FROM p
''').df().iloc[0]

kept_pages, kept_clients = len(frame), frame["client_hash_id"].nunique()
kept_impr = int(frame["impr_h1"].sum() + frame["impr_h2"].sum())
print(f"pages   kept: {kept_pages:>9,} of {int(lim.pages_with_search_data):,}  ({kept_pages / lim.pages_with_search_data:.0%})")
print(f"clients kept: {kept_clients:>9} of {int(lim.clients_with_search_data)}  ({kept_clients / lim.clients_with_search_data:.0%})")
print(f"impressions kept: {kept_impr / lim.impressions:.0%} of all March impressions")

share = frame.groupby("client_hash_id").size().sort_values(ascending=False)
print(f"largest 5 clients hold {share.head(5).sum() / share.sum():.0%} of the kept pages")

pages   kept:    63,489 of 176,738  (36%)
clients kept:        34 of 47  (72%)
impressions kept: 95% of all March impressions
largest 5 clients hold 76% of the kept pages
